# avoidZoneTest v4 (260731) — 단계 6-γ: 현실적 인접 OD 200개 검증 (군구 매핑 수정판)

## 이 노트북의 위치

| 버전 | 목적 | 결과 |
| --- | --- | --- |
| v3 (260727) | 무작위 OD 100쌍, γ 스윕 | hard case 70.8%. 단 O·D가 회피 영역 내부인 표본 다수 |
| v4 (260729) | O·D를 회피 영역 외부로 이동한 현실 표본 200쌍 | hard case 70.8%로 동일. 단 **남동구 누락 버그** 발견 |
| **v4 (260731, 본 노트북)** | 군구 매핑 버그 수정 후 재실행 | 9개 군구 전체 포함한 완전한 결과 확보 |

## v4(260729)에서 확인된 두 가지

**첫째, 표본 조건은 의도대로 개선되었다.** O·D가 회피 영역 내부인 OD 0개, 직선 미통과 0개,
이격 미달 0개로 생성 조건 위반이 없었다.

**둘째, 그럼에도 hard case 비율은 70.8%로 v3와 동일하였다.** v3는 34/48(70.83%),
v4는 131/185(70.81%)이다. 표본을 현실화해도 결과가 바뀌지 않았다는 것은,
hard case의 원인이 표본 설계가 아니라 **도로 네트워크 제약**임을 의미한다.
경로 중 회피 영역 통과 비율도 30.9%에서 26.8%로 4.1%p 감소에 그쳤다.

부수적으로 확인된 유효 지표는 OD 직선의 회피 영역 통과 길이(`chord_m`)이다.
이 값이 클수록 hard case 확률이 단조 증가하며, 2km 초과 구간에서는 100%였다.

| 직선 통과 길이 | OD 수 | hard case 비율 |
| --- | --- | --- |
| 500m 미만 | 29 | 38% |
| 500m~1km | 48 | 50% |
| 1~2km | 68 | 82% |
| 2km 초과 | 40 | 100% |

## 본 노트북에서 수정한 것

v4(260729)의 군구 매핑에서 **`'동구'`가 `'남동구'`의 부분 문자열**이라는 이유로
남동구 폴리곤이 전부 동구로 오분류되었다.

```python
# 버그: 딕셔너리 순회에서 donggu가 namdong보다 먼저 검사된다
for d_eng, kor in INCHEON_DISTRICT_KOR.items():
    if kor in str(p['sigg_name']):   # '동구' in '인천광역시 남동구' → True
```

이로 인해 두 가지 문제가 발생하였다. 남동구가 표본에서 완전히 제외되어 대상 군구가
9개가 아닌 8개였고, 동구로 분류된 OD 중 일부는 남동구 폴리곤 주변에 생성되었으나
핫스팟 후보는 동구에서 추출되어 지리적으로 어긋났다. 동구는 OD 25개 중 17개만
성공(8개 실패)하였고 hard case 비율이 100%로 가장 높았다.

수정 내용은 다음과 같다.

- **셀 3**: `sigg_name` 원본 문자열 형식을 출력하고, 긴 이름을 우선 검사하는
  `match_district()` 함수를 정의한다. 매핑 결과를 군구별로 집계하여 남동구가
  정상 분류되는지 검증한다.
- **셀 15**: `match_district()`를 사용하도록 폴리곤 분류 로직을 교체하고,
  군구별 폴리곤 수와 핫스팟 수를 함께 출력한다.

## 재실행에 관한 사항

기존 `output4_260729/realistic_od_200.pkl`은 **재사용하지 않는다.** 해당 표본은
버그가 있는 매핑으로 생성되어 남동구가 누락된 상태이므로, 재사용하면 수정의 의미가 없다.
OD 생성은 Tmap API를 호출하지 않으므로(기하 연산만 수행, 약 7초) 재생성에 따른
API 비용은 발생하지 않는다. 다만 OD가 바뀌면 배치 checkpoint도 재사용할 수 없어
Tmap 약 900~1,000회 호출과 6~7분의 실행 시간이 소요된다.

산출물은 `output4_260731/`에 저장하며, `output4_260729/`는 비교용으로 보존한다.

## v3 대비 실험 설계 (v4 공통)

| 항목 | v3 (단계 6-β) | v4 (단계 6-γ) |
| --- | --- | --- |
| O·D 위치 | 회피 폴리곤 중심 기준 양방향 (내부 배치 다수) | **모든 폴리곤 외부** + 이격 확보 |
| 직선 조건 | 인접/분리 사후 분류 | **직선이 폴리곤을 통과**하도록 강제 |
| 이격 거리 | 조건 없음 | **100 / 300 / 500m 균등 배분** |
| 폴리곤 선택 | 면적 가중 (미추홀 16/48 편중) | **군구 균등 순회** (본 판에서 9개 군구 전체) |
| γ | 0.5 / 1.0 / 2.0 / 5.0 스윕 | **1.0 고정** (v3에서 확정) |
| 표본 규모 | 인접 50 + 분리 50 | **현실적 인접 200** |
| 신규 기록 지표 | 없음 | 경로 통과 비율, 실제 이격, 직선 통과 길이 |

γ 고정 근거는 v3 결과이다. γ를 10배 변경하였음에도 인접 48쌍 중 41쌍(85%)의 결과가
동일하였고 총 감소율 차이는 2.6%p였다. 회피 영역 내부 핫스팟은 `avoid_penalty=999`로
γ와 무관하게 배제되므로 조정 여지가 구조적으로 제한된다.

## 변경하지 않는 것

셀 1~14의 알고리즘 함수는 v2/v3에서 그대로 계승한다(셀 3의 진단 블록만 추가).
알고리즘이 동일해야 결과 차이를 표본 차이로 귀속할 수 있다. pkl 영속화,
배치 checkpoint·resume, Tmap 응답 캐시 구조도 유지한다.

## 실행 순서

셀 1~14 → 셀 15(OD 200쌍 생성·검증) → 셀 16(배치 실행) → 셀 17(집계·v3 비교·환경 변수)
→ 셀 18(차트 6종) → 셀 19(대표 OD 지도)

셀 3 실행 시 `[군구 매핑 검증]`에 남동구가 표시되는지, 셀 15 실행 시
`[폴리곤 군구 분류]`가 9개 군구를 출력하는지 먼저 확인한다.


# Tmap API + 회피 알고리즘 재설계

In [ ]:
# 셀 1: 라이브러리 import 및 환경 점검
# v2 노트북 시작 — Tmap 도보 polyline 기반 회피 영역 검증

# 표준 라이브러리
import os
import json
import time
import math
import itertools
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

# HTTP·환경
import requests
from dotenv import load_dotenv

# 데이터 처리
import numpy as np
import pandas as pd

# 거리·공간 처리
from shapely.geometry import shape, Point, LineString
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.neighbors import BallTree   # 단계 6에서 neighbor_200m 계산용

# 지도 시각화
import folium

print(f"[v2 노트북 시작 — Tmap 도보 polyline 기반]")
print(f"  pandas    {pd.__version__}")
print(f"  numpy     {np.__version__}")
print(f"  folium    {folium.__version__}")
print(f"  requests  {requests.__version__}")
print(f"  작업 디렉토리: {os.getcwd()}")

In [ ]:
# 셀 2 (수정): 인증키 로드 + 출력 디렉토리를 output2로 분리
# v1 산출물(./output)과 v2 산출물(./output2)을 명확히 구분

load_dotenv(override=True)

VWORLD_KEY = os.getenv('VWORLD_KEY')
TMAP_APP_KEY = os.getenv('TMAP_APP_KEY')

if not VWORLD_KEY:
    raise RuntimeError(".env에 VWORLD_KEY가 없다.")
if not TMAP_APP_KEY:
    raise RuntimeError(".env에 TMAP_APP_KEY가 없다.")

print(f"[인증키 로드 완료]")
print(f"  V-World:  {VWORLD_KEY[:4]}...{VWORLD_KEY[-4:]} ({len(VWORLD_KEY)}자)")
print(f"  Tmap:     {TMAP_APP_KEY[:6]}...{TMAP_APP_KEY[-4:]} ({len(TMAP_APP_KEY)}자)")

# 출력 디렉토리 — v2 전용 폴더로 분리 (v1의 ./output과 공존)
OUTPUT_DIR = Path("./output2")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"  출력 디렉토리: {OUTPUT_DIR.resolve()}")
print(f"  (v1 산출물은 ./output에 그대로 보존)")

In [ ]:
# 셀 3: v1 회피 영역 GeoJSON 로드 + shapely 변환
# v1 노트북에서 V-World로 수집한 147개 폴리곤을 그대로 활용 (시간 절약)

from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.prepared import prep

# v1 산출물 경로 (output 폴더에 있음)
GEOJSON_PATH = Path("./output/incheon_avoid_zones_v1_260601.geojson")

if not GEOJSON_PATH.exists():
    raise FileNotFoundError(
        f"v1 GeoJSON 파일이 없다: {GEOJSON_PATH.resolve()}\n"
        f"v1 노트북의 셀 11까지 실행한 산출물이 있어야 한다."
    )

with open(GEOJSON_PATH, 'r', encoding='utf-8') as f:
    avoid_geojson = json.load(f)

# 147개 features → shapely 폴리곤 + 메타데이터
avoid_polygons = []
for feature in avoid_geojson['features']:
    props = feature['properties']
    geom = shape(feature['geometry'])
    avoid_polygons.append({
        'geom':       geom,
        'uqa_code':   props.get('uqa_code'),
        'uname':      props.get('uname'),
        'sigg_name':  props.get('sigg_name'),
        'feature_id': props.get('feature_id'),
    })

# 전체 폴리곤 union (교차 검사·페널티 계산용)
avoid_union = unary_union([p['geom'] for p in avoid_polygons])
avoid_union_prepared = prep(avoid_union)  # 점-폴리곤 검사 가속

# UQA 코드별 union (분석용)
avoid_union_by_code = {}
for code in ['UQA310', 'UQA320', 'UQA330']:
    geoms = [p['geom'] for p in avoid_polygons if p['uqa_code'] == code]
    if geoms:
        avoid_union_by_code[code] = unary_union(geoms)

# 통계 출력
print(f"[회피 영역 로드 완료]")
print(f"  총 폴리곤: {len(avoid_polygons)}개")
print(f"  UQA 코드별:")
for code in ['UQA310', 'UQA320', 'UQA330']:
    n = sum(1 for p in avoid_polygons if p['uqa_code'] == code)
    print(f"    {code}: {n}개")

print(f"\n  군구별:")
sigg_counts = Counter(p['sigg_name'] for p in avoid_polygons)
for sigg, n in sigg_counts.most_common():
    print(f"    {sigg:<15} {n}개")

print(f"\n  avoid_union bounds (lon_min, lat_min, lon_max, lat_max):")
print(f"    {avoid_union.bounds}")

# === v4(260731) 추가: sigg_name 원본 형식 확인 + 군구 매핑 검증 ===
# v4(260729)에서 '동구'가 '남동구'의 부분 문자열이라는 이유로 남동구 폴리곤이
# 동구로 오분류되었다. 그 결과 남동구가 표본에서 누락되고 동구 표본이 오염되었다.
# 부분 문자열 충돌을 막기 위해 긴 이름을 우선 검사하는 매핑 함수를 정의한다.

print("\n[sigg_name 원본 형식]")
for s in sorted(sigg_counts):
    print(f"    {repr(s)}")

DISTRICT_MATCH_ORDER = [
    ('michuhol', ['미추홀구', '미추홀']),
    ('namdong',  ['남동구',  '남동']),
    ('bupyeong', ['부평구',  '부평']),
    ('gyeyang',  ['계양구',  '계양']),
    ('yeonsu',   ['연수구',  '연수']),
    ('ganghwa',  ['강화군',  '강화']),
    ('seogu',    ['서구']),
    ('junggu',   ['중구']),
    ('donggu',   ['동구']),     # '남동구'보다 반드시 뒤에서 검사되어야 한다
]

def match_district(sigg_name):
    """sigg_name 문자열을 군구 영문 키로 변환한다 (부분 문자열 충돌 방지 순서)."""
    s = str(sigg_name)
    for d_eng, keys in DISTRICT_MATCH_ORDER:
        for k in keys:
            if k in s:
                return d_eng
    return None

print("\n[군구 매핑 검증]")
_map_counts = Counter()
_unmatched = []
for p in avoid_polygons:
    d = match_district(p['sigg_name'])
    if d is None:
        _unmatched.append(str(p['sigg_name']))
    else:
        _map_counts[d] += 1
for d, n in sorted(_map_counts.items(), key=lambda kv: -kv[1]):
    print(f"    {d:<10} {n}개")
print(f"    매칭 실패: {len(_unmatched)}개" + (f" {sorted(set(_unmatched))}" if _unmatched else ""))
if 'namdong' in _map_counts:
    print(f"    남동구 매핑 정상 ({_map_counts['namdong']}개) — v4_260729 오분류 해소")
else:
    print("    [경고] 남동구 폴리곤이 매핑되지 않았다. DISTRICT_MATCH_ORDER 점검 필요.")


In [ ]:
# 셀 4: 핫스팟 CSV 로드 + neighbor_200m + S_i 계산
# routeTest_v3 셀 3·5·6 이식 (핸드오프 v4 알고리즘 사양 그대로)

import ast
from sklearn.neighbors import BallTree

# === 1. CSV 로드 ===
CSV_PATH = Path("./hotspots_CCTV_20260510.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"핫스팟 CSV가 없다: {CSV_PATH.resolve()}")

hotspots = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

# list 필드 복원 (문자열 → list)
def parse_list_field(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return None
    return value

hotspots['install_years']      = hotspots['install_years'].apply(parse_list_field)
hotspots['relocation_history'] = hotspots['relocation_history'].apply(parse_list_field)

print(f"[핫스팟 로드] {len(hotspots):,}곳, {len(hotspots.columns)}컬럼")

# === 2. neighbor_200m 계산 (BallTree) ===
# 핸드오프 D2-2 공식: S_i = cctv_count + 0.5 × neighbor_200m
EARTH_RADIUS_KM = 6371.0

t0 = time.time()
coords_rad = np.radians(hotspots[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')
radius_rad = 0.2 / EARTH_RADIUS_KM   # 200m → 라디안

# query_radius는 자기 자신 포함 카운트 반환 → -1 보정
counts = tree.query_radius(coords_rad, r=radius_rad, count_only=True)
hotspots['neighbor_200m'] = counts - 1

print(f"\n[neighbor_200m 계산] 소요 {time.time() - t0:.2f}초")
print(f"  분포: min {hotspots['neighbor_200m'].min()}, "
      f"max {hotspots['neighbor_200m'].max()}, "
      f"평균 {hotspots['neighbor_200m'].mean():.2f}")

n_with_neighbor = (hotspots['neighbor_200m'] > 0).sum()
print(f"  200m 내 이웃 보유: {n_with_neighbor:,}곳 ({n_with_neighbor/len(hotspots)*100:.1f}%)")

# === 3. S_i 계산 ===
hotspots['S_i'] = hotspots['cctv_count'] + 0.5 * hotspots['neighbor_200m']

print(f"\n[S_i 계산 완료]")
print(f"  분포: min {hotspots['S_i'].min():.2f}, "
      f"max {hotspots['S_i'].max():.2f}, "
      f"평균 {hotspots['S_i'].mean():.2f}")

print(f"\n  군구별 S_i 평균:")
for sigg, mean_si in hotspots.groupby('source_district')['S_i'].mean().items():
    print(f"    {sigg:<12} {mean_si:.2f}")

# === 4. 회피 영역 내부 핫스팟 표시 (방법 C에서 활용) ===
# 각 핫스팟이 회피 영역 내부에 있는지 미리 계산
def is_in_avoid_zone(row):
    pt = Point(row['longitude'], row['latitude'])
    return avoid_union_prepared.contains(pt)

t0 = time.time()
hotspots['in_avoid_zone'] = hotspots.apply(is_in_avoid_zone, axis=1)
print(f"\n[회피 영역 내부 검사] 소요 {time.time() - t0:.2f}초")
print(f"  회피 영역 내부 핫스팟: {hotspots['in_avoid_zone'].sum()}/{len(hotspots):,} "
      f"({hotspots['in_avoid_zone'].sum()/len(hotspots)*100:.2f}%)")

In [ ]:
# 셀 5: 거리·Buffer 유틸 함수 + OD 9쌍 + 후보 추출 검증
# haversine 패키지 의존 제거 — numpy 기반 직접 구현 (수학적으로 동일)

# === 0. Haversine 직접 구현 ===
def haversine_km(p_a, p_b):
    """두 (lat, lon) 좌표 사이의 Haversine 거리 (km)
    
    haversine 패키지의 hs.haversine(p_a, p_b, unit=hs.Unit.KILOMETERS)와 동일 결과.
    지구 반경 6371.0 km 사용.
    """
    lat1, lon1 = p_a
    lat2, lon2 = p_b
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return EARTH_RADIUS_KM * c   # EARTH_RADIUS_KM은 셀 4에서 6371.0으로 정의됨

# === 1. 도보·Buffer 유틸 함수 ===
DETOUR_RATIO = 1.3   # 한국 도시 도보 환경 평균 보정계수 (핸드오프 D3)

def walking_distance_km(p_a, p_b):
    """추정 도보 거리 (km) = Haversine × 1.3"""
    return haversine_km(p_a, p_b) * DETOUR_RATIO

def detour_distance_km(origin, hotspot, destination):
    """핫스팟 단독 경유 시 우회 거리: D_i = d(O,h) + d(h,D) - d(O,D)"""
    d_oh = walking_distance_km(origin, hotspot)
    d_hd = walking_distance_km(hotspot, destination)
    d_od = walking_distance_km(origin, destination)
    return max(d_oh + d_hd - d_od, 0.0)

def get_buffer_radius(district):
    """군구별 buffer 반경 (km) — 강화군 2.5km, 그 외 1.5km (핸드오프 D4)"""
    return 2.5 if district == 'ganghwa' else 1.5

def filter_candidates_by_ellipse_buffer(df, origin, destination, buffer_r_km):
    """타원 buffer 필터링: d(O,h) + d(h,D) - d(O,D) < buffer_r × 1.5"""
    d_od = walking_distance_km(origin, destination)
    threshold = buffer_r_km * 1.5
    
    coords = df[['latitude', 'longitude']].values
    d_oh = np.array([walking_distance_km(origin, tuple(c)) for c in coords])
    d_hd = np.array([walking_distance_km(tuple(c), destination) for c in coords])
    
    mask = (d_oh + d_hd - d_od) < threshold
    return df[mask].copy()

# === 2. OD 9쌍 좌표 (routeTest_v3 OD_CANDIDATES_V2와 동일) ===
OD_CANDIDATES_V2 = {
    'ganghwa':  {'O': (37.706505, 126.443398), 'D': (37.732838, 126.517697)},
    'junggu':   {'O': (37.473565, 126.621778), 'D': (37.474748, 126.603258)},
    'donggu':   {'O': (37.470733, 126.641914), 'D': (37.487780, 126.621754)},
    'michuhol': {'O': (37.455015, 126.667435), 'D': (37.435062, 126.690721)},
    'seogu':    {'O': (37.545115, 126.676062), 'D': (37.552748, 126.650239)},
    'yeonsu':   {'O': (37.391182, 126.645092), 'D': (37.418275, 126.661122)},
    'namdong':  {'O': (37.447245, 126.731453), 'D': (37.447769, 126.700089)},
    'bupyeong': {'O': (37.507046, 126.721858), 'D': (37.482179, 126.703038)},
    'gyeyang':  {'O': (37.537367, 126.737744), 'D': (37.530279, 126.722516)},
}

# === 3. 각 OD에 대해 Buffer 후보 추출 + 표본 편향 검증 ===
print(f"[Buffer 후보 추출 검증]")
print(f"{'군구':<10} {'직선(km)':>9} {'buffer(km)':>11} {'후보수':>7} {'회피_내부':>9}")
print("-" * 55)

od_buffer_results = {}
for district, od in OD_CANDIDATES_V2.items():
    origin = od['O']
    destination = od['D']
    
    direct_km = haversine_km(origin, destination)
    buffer_r = get_buffer_radius(district)
    
    district_df = hotspots[hotspots['source_district'] == district]
    cands = filter_candidates_by_ellipse_buffer(district_df, origin, destination, buffer_r)
    n_avoid_in_buffer = int(cands['in_avoid_zone'].sum())
    
    od_buffer_results[district] = {
        'origin': origin,
        'destination': destination,
        'direct_km': direct_km,
        'buffer_r_km': buffer_r,
        'candidates': cands,
        'n_candidates': len(cands),
        'n_avoid_in_buffer': n_avoid_in_buffer,
    }
    
    print(f"{district:<10} {direct_km:>7.2f}   {buffer_r:>9}   {len(cands):>5}   "
          f"{n_avoid_in_buffer:>6}")

print(f"\n[중요 — 표본 편향 가설 검증]")
total_avoid_in_buffer = sum(r['n_avoid_in_buffer'] for r in od_buffer_results.values())
print(f"  9 OD의 Buffer 안 회피 핫스팟 총합: {total_avoid_in_buffer}곳")
if total_avoid_in_buffer == 0:
    print(f"  → 표본 편향 가설 정량 입증: Buffer 안에 회피 핫스팟이 0건이므로")
    print(f"    옵션 X 0건은 알고리즘 견고성이 아닌 OD 선정 편향의 결과")
elif total_avoid_in_buffer < 5:
    print(f"  → 표본 편향 가설 부분 입증: Buffer 안 회피 핫스팟이 매우 적음")
else:
    print(f"  → 알고리즘이 일부 회피 효과 보일 가능성 (S_i 정렬에서 탈락 등)")

## 정적 회피 페널티 함수 (방법 C 핵심)

In [ ]:
# 셀 6: 정적 회피 페널티 함수 (방법 C 핵심)
# Tmap 호출 없이 shapely 직선 교차로 회피 페널티 계산
# 위경도 1도 ≈ 111km (위도 37도 부근) 근사 사용

def compute_avoid_penalty(hotspot_lat, hotspot_lon, origin, destination,
                          avoid_union, avoid_union_prepared):
    """
    핫스팟의 정적 회피 페널티 계산.
    
    구성 요소:
    1. 핫스팟이 회피 영역 내부면 매우 큰 페널티 (999) — 옵션 X 완전 차단
    2. O → 핫스팟 직선의 회피 영역 통과 길이
    3. 핫스팟 → D 직선의 회피 영역 통과 길이
    
    단위: km (다른 항과 일치)
    호출 비용: O(1), 약 1~2 밀리초
    
    Args:
        hotspot_lat, hotspot_lon: 핫스팟 좌표
        origin: (lat, lon) 출발지
        destination: (lat, lon) 도착지
        avoid_union: shapely MultiPolygon (회피 영역 union)
        avoid_union_prepared: prep된 union (점-폴리곤 검사 가속)
    
    Returns:
        float: 페널티 (km)
    """
    # 1. 회피 영역 내부 검사 (옵션 X 차단)
    hotspot_point = Point(hotspot_lon, hotspot_lat)
    if avoid_union_prepared.contains(hotspot_point):
        return 999.0
    
    # 2. O → 핫스팟 직선의 회피 영역 통과
    line_o = LineString([
        (origin[1], origin[0]),
        (hotspot_lon, hotspot_lat),
    ])
    inter_o = line_o.intersection(avoid_union)
    len_o_deg = inter_o.length if not inter_o.is_empty else 0
    
    # 3. 핫스팟 → D 직선의 회피 영역 통과
    line_d = LineString([
        (hotspot_lon, hotspot_lat),
        (destination[1], destination[0]),
    ])
    inter_d = line_d.intersection(avoid_union)
    len_d_deg = inter_d.length if not inter_d.is_empty else 0
    
    # 위경도 → km 근사 (위도 37도 부근 1도 ≈ 111km)
    return (len_o_deg + len_d_deg) * 111.0


# === 동작 검증 — donggu·gyeyang의 회피 후보 vs 일반 후보 페널티 비교 ===
print("[정적 회피 페널티 함수 검증]")
print("=" * 70)

for district in ['donggu', 'gyeyang', 'michuhol']:   # 회피 7곳·7곳·0곳 비교
    od = OD_CANDIDATES_V2[district]
    cands = od_buffer_results[district]['candidates']
    
    # 회피 영역 내부 후보의 페널티
    avoid_cands = cands[cands['in_avoid_zone']]
    
    print(f"\n[{district}] OD: O{od['O']} → D{od['D']}")
    print(f"  Buffer 후보 {len(cands)}곳 중 회피 내부 {len(avoid_cands)}곳")
    
    if len(avoid_cands) > 0:
        print(f"\n  회피 영역 내부 후보 (S_i 상위 3개):")
        for _, row in avoid_cands.nlargest(3, 'S_i').iterrows():
            penalty = compute_avoid_penalty(
                row['latitude'], row['longitude'],
                od['O'], od['D'],
                avoid_union, avoid_union_prepared,
            )
            print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
                  f"penalty={penalty:.2f}")
    
    # 비교: 회피 영역 밖 후보 상위 3
    non_avoid = cands[~cands['in_avoid_zone']]
    print(f"\n  회피 영역 밖 후보 (S_i 상위 3개):")
    for _, row in non_avoid.nlargest(3, 'S_i').iterrows():
        penalty = compute_avoid_penalty(
            row['latitude'], row['longitude'],
            od['O'], od['D'],
            avoid_union, avoid_union_prepared,
        )
        print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
              f"penalty={penalty:.2f}")

In [ ]:
# 셀 7: Score 공식 통합 — 방법 C 적용
# 새 공식: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i

# === 하이퍼파라미터 ===
ALPHA = 0.5   # S_i 가중치 (핸드오프 D6 그대로)
BETA  = 0.5   # D_i 가중치 (핸드오프 D6 그대로)
GAMMA = 1.0   # avoid_penalty 가중치 (방법 C 신규, 단계 6 후반 튜닝 대상)

# === 회피 페널티 통합 Score 함수 (방법 C) ===
def compute_scores_with_avoidance(df, origin, destination,
                                   avoid_union, avoid_union_prepared,
                                   alpha=ALPHA, beta=BETA, gamma=GAMMA):
    """방법 C 적용 Score 계산.
    Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
    """
    out = df.copy()
    out['D_i'] = out.apply(
        lambda r: detour_distance_km(origin, (r['latitude'], r['longitude']), destination),
        axis=1
    )
    out['avoid_penalty_i'] = out.apply(
        lambda r: compute_avoid_penalty(
            r['latitude'], r['longitude'],
            origin, destination,
            avoid_union, avoid_union_prepared,
        ),
        axis=1
    )
    out['Score_i'] = alpha * out['S_i'] - beta * out['D_i'] - gamma * out['avoid_penalty_i']
    return out

# === 원본 Score 함수 (방법 C 적용 전, 회귀 검증용) ===
def compute_scores_original(df, origin, destination, alpha=ALPHA, beta=BETA):
    """routeTest_v3 원본 Score (회귀 검증용)."""
    out = df.copy()
    out['D_i'] = out.apply(
        lambda r: detour_distance_km(origin, (r['latitude'], r['longitude']), destination),
        axis=1
    )
    out['avoid_penalty_i'] = 0.0   # 없음 (호환성)
    out['Score_i'] = alpha * out['S_i'] - beta * out['D_i']
    return out

print(f"[Score 공식 정의 완료]")
print(f"  원본: Score_i = α·S_i - β·D_i")
print(f"  방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i")
print(f"  가중치: α={ALPHA}, β={BETA}, γ={GAMMA} (초기값)")
print(f"  γ 튜닝은 단계 6-D에서 진행 (0.5, 1.0, 2.0, 5.0 비교)")

In [ ]:
# 셀 8: Greedy 알고리즘 + 방법 C vs 원본 비교
# routeTest_v3의 algorithm_greedy 이식 + 회피 페널티 적용 버전 추가

K = 3
BF_TOP_N = 15

def algorithm_greedy_v2(candidates_df, origin, destination,
                         avoid_union, avoid_union_prepared,
                         alpha=ALPHA, beta=BETA, gamma=GAMMA, k=K):
    """방법 C 적용 Greedy.
    1. S_i 상위 BF_TOP_N=15 풀 (원본과 동일)
    2. compute_scores_with_avoidance로 Score_i 계산 (회피 페널티 포함)
    3. Score_i 상위 K=3 선택
    """
    t0 = time.time()
    n = len(candidates_df)
    if n == 0:
        return pd.DataFrame(), 0
    k_eff = min(k, n)
    
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    scored = compute_scores_with_avoidance(
        pool, origin, destination, avoid_union, avoid_union_prepared,
        alpha=alpha, beta=beta, gamma=gamma,
    )
    selected = scored.nlargest(k_eff, 'Score_i')
    return selected, (time.time() - t0) * 1000

def algorithm_greedy_original(candidates_df, origin, destination,
                               alpha=ALPHA, beta=BETA, k=K):
    """routeTest_v3 원본 Greedy (회귀 검증용)."""
    t0 = time.time()
    n = len(candidates_df)
    if n == 0:
        return pd.DataFrame(), 0
    k_eff = min(k, n)
    
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    scored = compute_scores_original(pool, origin, destination, alpha=alpha, beta=beta)
    selected = scored.nlargest(k_eff, 'Score_i')
    return selected, (time.time() - t0) * 1000


# === 9 OD에 대해 원본 vs 방법 C 직접 비교 ===
print("=" * 80)
print("[Greedy 원본 vs 방법 C 비교 — 9 OD]")
print("=" * 80)

comparison_results = []
for district, od in OD_CANDIDATES_V2.items():
    cands = od_buffer_results[district]['candidates']
    
    # 원본
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    orig_ids = set(sel_orig['hotspot_id'])
    orig_avoid_count = int(sel_orig['in_avoid_zone'].sum()) if 'in_avoid_zone' in sel_orig.columns else 0
    
    # 방법 C
    sel_v2, _ = algorithm_greedy_v2(cands, od['O'], od['D'], avoid_union, avoid_union_prepared)
    v2_ids = set(sel_v2['hotspot_id'])
    v2_avoid_count = int(sel_v2['in_avoid_zone'].sum()) if 'in_avoid_zone' in sel_v2.columns else 0
    
    # K=3 일치 여부
    if orig_ids == v2_ids:
        diff_status = "동일"
    else:
        diff_status = f"다름 (원본만: {len(orig_ids - v2_ids)}, v2만: {len(v2_ids - orig_ids)})"
    
    # 방법 C의 페널티 합계
    v2_total_penalty = sel_v2['avoid_penalty_i'].sum()
    
    comparison_results.append({
        'district': district,
        'orig_avoid_in_K3': orig_avoid_count,
        'v2_avoid_in_K3': v2_avoid_count,
        'diff_status': diff_status,
        'v2_total_penalty': v2_total_penalty,
    })
    
    print(f"\n[{district}]")
    print(f"  원본 K=3 회피 영역 내부: {orig_avoid_count}개")
    print(f"  방법 C K=3 회피 영역 내부: {v2_avoid_count}개")
    print(f"  K=3 비교: {diff_status}")
    print(f"  방법 C 총 페널티: {v2_total_penalty:.2f}")

# 종합
print(f"\n{'=' * 80}")
print(f"[종합]")
df_compare = pd.DataFrame(comparison_results)
n_diff = sum(1 for r in comparison_results if "다름" in r['diff_status'])
print(f"  K=3 선정이 다른 OD: {n_diff}/9")
print(f"  원본 K=3 회피 내부 총: {df_compare['orig_avoid_in_K3'].sum()}")
print(f"  방법 C K=3 회피 내부 총: {df_compare['v2_avoid_in_K3'].sum()} (페널티 999로 차단)")

### Tmap 보행자 호출 + polyline 교차 검사

In [ ]:
# 셀 9: Tmap 보행자 호출 함수 + polyline 교차 검사
# v1 노트북의 셀 21을 v2에 이식 + 교차 검사 함수 추가

TMAP_PEDESTRIAN_URL = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

def get_pedestrian_route(origin, destination, waypoints=None, debug=False):
    """Tmap 보행자 경로안내 API 호출.
    
    Args:
        origin, destination: (lat, lon)
        waypoints: [(lat, lon), ...] 또는 None (최대 5개)
    Returns:
        dict: path([(lat, lon), ...]), distance_m, duration_ms, status
    """
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "appKey": TMAP_APP_KEY,
    }
    payload = {
        "startX": origin[1], "startY": origin[0],
        "endX": destination[1], "endY": destination[0],
        "startName": "출발지", "endName": "도착지",
        "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",
        "searchOption": "0",
    }
    if waypoints:
        if len(waypoints) > 5:
            return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                    'status': f'ERROR: 경유지 {len(waypoints)}개. Tmap 한도 5개 초과'}
        payload["passList"] = "_".join(f"{wp[1]},{wp[0]}" for wp in waypoints)
    
    try:
        r = requests.post(TMAP_PEDESTRIAN_URL, headers=headers, json=payload, timeout=15)
    except requests.exceptions.RequestException as e:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: {type(e).__name__}'}
    
    if r.status_code != 200:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: HTTP {r.status_code}, body: {r.text[:150]}'}
    
    try:
        result = r.json()
    except json.JSONDecodeError:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'JSON_ERROR'}
    
    if result.get('type') != 'FeatureCollection':
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'INVALID'}
    
    features = result.get('features', [])
    if not features:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'NO_FEATURES'}
    
    first_props = features[0].get('properties', {})
    total_distance_m = first_props.get('totalDistance', 0)
    total_time_s = first_props.get('totalTime', 0)
    
    path_lonlat = []
    for f in features:
        geom = f.get('geometry', {})
        if geom.get('type') == 'LineString':
            path_lonlat.extend(geom.get('coordinates', []))
    
    # (lat, lon) 형식으로 변환 (본 노트북 일관성)
    path_latlon = [(pt[1], pt[0]) for pt in path_lonlat]
    
    return {
        'path': path_latlon,
        'distance_m': total_distance_m,
        'duration_ms': total_time_s * 1000,
        'status': 'OK',
    }


def check_polyline_avoid_intersection(path_latlon, avoid_union_param, 
                                        avoid_union_by_code_param):
    """polyline이 회피 영역과 교차하는지 검사.
    
    Args:
        path_latlon: [(lat, lon), ...] polyline 좌표 리스트
        avoid_union_param: shapely MultiPolygon (전체 회피 영역)
        avoid_union_by_code_param: dict {uqa_code: union} (코드별 회피 영역)
    
    Returns:
        dict: crosses, intersection_length_m, intersection_ratio, crossed_uqa_codes
    """
    empty_result = {
        'crosses': False, 'intersection_length_m': 0, 
        'intersection_ratio': 0, 'crossed_uqa_codes': [],
    }
    
    if not path_latlon or len(path_latlon) < 2:
        return empty_result
    
    # polyline → shapely LineString (lon, lat 순서)
    line = LineString([(pt[1], pt[0]) for pt in path_latlon])
    
    if not line.intersects(avoid_union_param):
        return empty_result
    
    intersection = line.intersection(avoid_union_param)
    inter_length_m = intersection.length * 111000.0
    total_length_m = line.length * 111000.0
    ratio = inter_length_m / total_length_m if total_length_m > 0 else 0
    
    # 어느 UQA 코드와 교차?
    crossed_codes = []
    for code, union in avoid_union_by_code_param.items():
        if line.intersects(union):
            crossed_codes.append(code)
    
    return {
        'crosses': True,
        'intersection_length_m': inter_length_m,
        'intersection_ratio': ratio,
        'crossed_uqa_codes': crossed_codes,
    }


# === 동작 검증 — junggu·donggu에서 원본 K=3 vs 방법 C K=3 polyline 직접 비교 ===
print("[Tmap 보행자 호출 + 교차 검사 동작 검증]")
print("=" * 80)
print("방법 C가 K=3을 다르게 선택한 2개 OD에서 polyline 통과 효과 측정\n")

for district in ['junggu', 'donggu']:
    od = OD_CANDIDATES_V2[district]
    cands = od_buffer_results[district]['candidates']
    
    print(f"[{district}]")
    
    # 원본 K=3
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    wp_orig = [(row['latitude'], row['longitude']) 
               for _, row in sel_orig.iterrows()]
    
    # 방법 C K=3
    sel_v2, _ = algorithm_greedy_v2(cands, od['O'], od['D'],
                                      avoid_union, avoid_union_prepared)
    wp_v2 = [(row['latitude'], row['longitude'])
             for _, row in sel_v2.iterrows()]
    
    # 원본 polyline
    r_orig = get_pedestrian_route(od['O'], od['D'], waypoints=wp_orig)
    if r_orig['status'] == 'OK':
        c_orig = check_polyline_avoid_intersection(
            r_orig['path'], avoid_union, avoid_union_by_code)
        print(f"  원본 K=3 도보 polyline:")
        print(f"    거리 {r_orig['distance_m']/1000:.2f}km, 점 {len(r_orig['path'])}개")
        if c_orig['crosses']:
            print(f"    교차 {c_orig['intersection_length_m']:.0f}m "
                  f"({c_orig['intersection_ratio']*100:.2f}%), "
                  f"UQA {c_orig['crossed_uqa_codes']}")
        else:
            print(f"    교차 없음")
    else:
        print(f"  원본 polyline 호출 실패: {r_orig['status']}")
        continue
    
    # 방법 C polyline
    r_v2 = get_pedestrian_route(od['O'], od['D'], waypoints=wp_v2)
    if r_v2['status'] == 'OK':
        c_v2 = check_polyline_avoid_intersection(
            r_v2['path'], avoid_union, avoid_union_by_code)
        print(f"  방법 C K=3 도보 polyline:")
        print(f"    거리 {r_v2['distance_m']/1000:.2f}km, 점 {len(r_v2['path'])}개")
        if c_v2['crosses']:
            print(f"    교차 {c_v2['intersection_length_m']:.0f}m "
                  f"({c_v2['intersection_ratio']*100:.2f}%), "
                  f"UQA {c_v2['crossed_uqa_codes']}")
        else:
            print(f"    교차 없음")
        
        # 비교
        diff_m = c_v2['intersection_length_m'] - c_orig['intersection_length_m']
        if diff_m < 0:
            print(f"  방법 C 효과: 교차 {abs(diff_m):.0f}m 감소")
        elif diff_m > 0:
            print(f"  방법 C 효과: 교차 {diff_m:.0f}m 증가 (예상 외)")
        else:
            print(f"  방법 C 효과: 변화 없음")
    print()

In [ ]:
# 셀 10: swap 함수 — 방법 C K=3의 polyline이 통과 시 후보 교체

MAX_SWAP_ATTEMPTS = 5   # swap 최대 시도 횟수

def attempt_swap(scored_pool, initial_selection, origin, destination,
                  avoid_union_param, avoid_union_by_code_param,
                  max_attempts=MAX_SWAP_ATTEMPTS, verbose=False):
    """K=3 swap — polyline 통과 시 회피 페널티 가장 큰 핫스팟을 다음 후보로 교체.
    
    전략:
    1. 현재 K=3로 polyline 호출 + 교차 검사
    2. 교차 없으면 종료 (성공)
    3. 교차 시:
       a. 현재 K=3 중 avoid_penalty_i 가장 큰 후보 식별
       b. pool에서 K=3에 포함 안 된 다음 Score_i 순위 후보로 교체
       c. 새 K=3로 polyline 재호출
    4. max_attempts 초과 시 종료 (hard case)
    
    Args:
        scored_pool: compute_scores_with_avoidance 적용된 Top-15 풀
        initial_selection: 방법 C가 선정한 초기 K=3 DataFrame
        origin, destination: OD 좌표
        max_attempts: swap 시도 한도
    
    Returns:
        dict: final_selection, final_polyline, final_intersection, 
              final_distance_m, final_duration_ms, n_swaps, hard_case, swap_history
    """
    selected_ids = set(initial_selection['hotspot_id'])
    
    # 대체 후보: pool 중 초기 K=3에 안 들어간 후보를 Score_i 내림차순
    alternates = scored_pool[~scored_pool['hotspot_id'].isin(selected_ids)]
    alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
    swap_history = []
    
    for attempt in range(max_attempts + 1):
        # 현재 K=3로 polyline 호출
        waypoints = [(row['latitude'], row['longitude']) 
                     for _, row in current.iterrows()]
        route = get_pedestrian_route(origin, destination, waypoints=waypoints)
        
        if route['status'] != 'OK':
            return {
                'final_selection': current, 'final_polyline': None,
                'final_intersection': None, 'final_distance_m': 0,
                'final_duration_ms': 0, 'n_swaps': attempt,
                'hard_case': True, 'swap_history': swap_history,
                'error': route['status'],
            }
        
        intersection = check_polyline_avoid_intersection(
            route['path'], avoid_union_param, avoid_union_by_code_param)
        
        # 교차 없으면 성공
        if not intersection['crosses']:
            if verbose:
                print(f"  swap {attempt}회 후 교차 해소")
            return {
                'final_selection': current,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': False,
                'swap_history': swap_history,
            }
        
        # 교차 발생 → swap 시도
        if attempt >= max_attempts or len(alternates) == 0:
            # 더 이상 swap 못 함 → hard case
            if verbose:
                print(f"  swap {attempt}회 후 hard case (한도 도달 또는 대체 후보 소진)")
            return {
                'final_selection': current,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': True,
                'swap_history': swap_history,
            }
        
        # 현재 K=3 중 페널티 가장 큰 후보 식별
        worst_idx = current['avoid_penalty_i'].idxmax()
        worst_id = current.loc[worst_idx, 'hotspot_id']
        worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
        # 다음 후보로 교체
        new_candidate = alternates.iloc[0]
        alternates = alternates.iloc[1:].reset_index(drop=True)
        
        if verbose:
            print(f"  swap #{attempt+1}: {worst_id} (penalty {worst_penalty:.2f}) "
                  f"→ {new_candidate['hotspot_id']}")
        
        swap_history.append({
            'attempt': attempt + 1,
            'removed': worst_id,
            'removed_penalty': worst_penalty,
            'added': new_candidate['hotspot_id'],
            'added_penalty': new_candidate['avoid_penalty_i'],
            'intersection_before_m': intersection['intersection_length_m'],
        })
        
        # K=3 갱신
        current = current.drop(index=worst_idx).reset_index(drop=True)
        new_row = new_candidate.to_frame().T
        current = pd.concat([current, new_row], ignore_index=True)
        current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    # 도달 안 함
    return None


# === 동작 검증 — donggu에서 단독 swap 호출 ===
print("=" * 80)
print("[swap 함수 단독 동작 검증 — donggu]")
print("=" * 80)

district = 'donggu'
od = OD_CANDIDATES_V2[district]
cands = od_buffer_results[district]['candidates']

# 방법 C로 초기 K=3 선정
pool = cands.nlargest(min(BF_TOP_N, len(cands)), 'S_i')
scored_pool = compute_scores_with_avoidance(
    pool, od['O'], od['D'], avoid_union, avoid_union_prepared)
initial_sel = scored_pool.nlargest(K, 'Score_i')

print(f"  Buffer 후보: {len(cands)}곳 / S_i Top-{BF_TOP_N} 풀")
print(f"  초기 K=3 (방법 C 적용):")
for _, row in initial_sel.iterrows():
    print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
          f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}")

# swap 실행
print(f"\n  swap 시도 (verbose):")
swap_result = attempt_swap(
    scored_pool, initial_sel, od['O'], od['D'],
    avoid_union, avoid_union_by_code,
    verbose=True,
)

print(f"\n  최종 결과:")
print(f"    swap 횟수: {swap_result['n_swaps']}")
print(f"    hard case: {swap_result['hard_case']}")
if swap_result['final_intersection']:
    fi = swap_result['final_intersection']
    print(f"    최종 polyline 교차: {fi['intersection_length_m']:.0f}m "
          f"({fi['intersection_ratio']*100:.2f}%)")
print(f"    최종 도보 거리: {swap_result['final_distance_m']/1000:.2f}km")
print(f"    최종 K=3:")
for _, row in swap_result['final_selection'].iterrows():
    print(f"      {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
          f"penalty={row['avoid_penalty_i']:.2f}")

In [ ]:
# # 셀 11: 방법 C+swap 통합 알고리즘 함수 정의
# # visit_order는 Score_i 내림차순 고정 (evaluate_path 미사용)

# def algorithm_greedy_with_avoidance(candidates_df, origin, destination,
#                                       avoid_union_param, avoid_union_prepared_param,
#                                       avoid_union_by_code_param,
#                                       alpha=ALPHA, beta=BETA, gamma=GAMMA,
#                                       k=K, max_swap=MAX_SWAP_ATTEMPTS,
#                                       verbose=False):
#     """방법 C + swap 통합 알고리즘 (4번째 알고리즘 후보 E_v2).
    
#     파이프라인:
#     1. Buffer 후보 → S_i Top-15 풀
#     2. 방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
#     3. Score_i Top-K=3 초기 선정
#     4. K=3을 Score_i 내림차순으로 방문 (visit_order 최적화 없음)
#     5. Tmap polyline 호출 + 교차 검사
#     6. 교차 시 swap (penalty 큰 후보를 다음 Score 후보로 교체)
#     7. 최대 max_swap회까지 반복 후 hard case 종료
    
#     Args:
#         candidates_df: Buffer 후보 DataFrame
#         origin, destination: (lat, lon)
#         avoid_*: 셀 3에서 만든 회피 영역 객체들
#         gamma: 회피 페널티 가중치 (기본 1.0)
#         max_swap: swap 최대 시도 (기본 5)
#         verbose: 진행 출력 여부
    
#     Returns:
#         dict: final_selection, visit_order, final_polyline, final_intersection,
#               final_distance_m, final_duration_ms, n_swaps, hard_case,
#               swap_history, total_elapsed_ms
#     """
#     t0 = time.time()
    
#     # 1. S_i Top-15 풀
#     n = len(candidates_df)
#     if n == 0:
#         return None
#     pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    
#     # 2. 방법 C: Score_i 계산
#     scored_pool = compute_scores_with_avoidance(
#         pool, origin, destination,
#         avoid_union_param, avoid_union_prepared_param,
#         alpha=alpha, beta=beta, gamma=gamma,
#     )
    
#     # 3. Score_i Top-K=3 초기 선정
#     k_eff = min(k, n)
#     initial_selection = scored_pool.nlargest(k_eff, 'Score_i')
    
#     # 대체 후보: pool 중 K=3에 없는 것, Score_i 내림차순
#     alternates = scored_pool[~scored_pool['hotspot_id'].isin(initial_selection['hotspot_id'])]
#     alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
#     # 현재 K=3 (Score_i 내림차순으로 정렬)
#     current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
#     swap_history = []
    
#     if verbose:
#         print(f"  초기 K=3 선정 완료. swap 루프 진입.")
    
#     # 4·5·6. polyline + swap 루프 (visit_order = Score_i 순서 고정)
#     for attempt in range(max_swap + 1):
#         # 4. K=3을 Score_i 내림차순(현재 정렬 상태)으로 방문
#         visit_order = current['hotspot_id'].tolist()
#         ordered_waypoints = [(row['latitude'], row['longitude'])
#                               for _, row in current.iterrows()]
        
#         # 5. Tmap polyline 호출
#         route = get_pedestrian_route(origin, destination, waypoints=ordered_waypoints)
#         if route['status'] != 'OK':
#             return {
#                 'final_selection': current, 'visit_order': visit_order,
#                 'final_polyline': None, 'final_intersection': None,
#                 'final_distance_m': 0, 'final_duration_ms': 0,
#                 'n_swaps': attempt, 'hard_case': True,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#                 'error': route['status'],
#             }
        
#         intersection = check_polyline_avoid_intersection(
#             route['path'], avoid_union_param, avoid_union_by_code_param)
        
#         if verbose:
#             cross_info = (f"{intersection['intersection_length_m']:.0f}m"
#                           if intersection['crosses'] else "없음")
#             print(f"  attempt {attempt}: 교차 {cross_info}")
        
#         # 6a. 교차 없으면 성공 종료
#         if not intersection['crosses']:
#             return {
#                 'final_selection': current,
#                 'visit_order': visit_order,
#                 'final_polyline': route['path'],
#                 'final_intersection': intersection,
#                 'final_distance_m': route['distance_m'],
#                 'final_duration_ms': route['duration_ms'],
#                 'n_swaps': attempt,
#                 'hard_case': False,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#             }
        
#         # 6b. swap 한도 도달 → hard case 종료
#         if attempt >= max_swap or len(alternates) == 0:
#             return {
#                 'final_selection': current,
#                 'visit_order': visit_order,
#                 'final_polyline': route['path'],
#                 'final_intersection': intersection,
#                 'final_distance_m': route['distance_m'],
#                 'final_duration_ms': route['duration_ms'],
#                 'n_swaps': attempt,
#                 'hard_case': True,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#             }
        
#         # 6c. swap 시도: penalty 가장 큰 후보 → 다음 Score 후보
#         worst_idx = current['avoid_penalty_i'].idxmax()
#         worst_id = current.loc[worst_idx, 'hotspot_id']
#         worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
#         new_candidate = alternates.iloc[0]
#         alternates = alternates.iloc[1:].reset_index(drop=True)
        
#         swap_history.append({
#             'attempt': attempt + 1,
#             'removed': worst_id,
#             'removed_penalty': worst_penalty,
#             'added': new_candidate['hotspot_id'],
#             'added_penalty': new_candidate['avoid_penalty_i'],
#             'intersection_before_m': intersection['intersection_length_m'],
#         })
        
#         # K=3 갱신 (Score_i 내림차순 재정렬)
#         current = current.drop(index=worst_idx).reset_index(drop=True)
#         new_row = new_candidate.to_frame().T
#         current = pd.concat([current, new_row], ignore_index=True)
#         current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
#     return None   # 도달 안 함


# print("[통합 알고리즘 함수 정의 완료]")
# print(f"  파이프라인: Buffer → S_i Top-{BF_TOP_N} → 방법 C Score_i Top-{K} → Tmap → swap")
# print(f"  visit_order: Score_i 내림차순 (최적화 없음)")
# print(f"  하이퍼파라미터: α={ALPHA}, β={BETA}, γ={GAMMA}, K={K}, max_swap={MAX_SWAP_ATTEMPTS}")


# # === 단독 동작 검증 — junggu ===
# print(f"\n{'=' * 80}")
# print(f"[통합 알고리즘 단독 검증 — junggu]")
# print(f"{'=' * 80}")

# district = 'junggu'
# od = OD_CANDIDATES_V2[district]
# cands = od_buffer_results[district]['candidates']

# result = algorithm_greedy_with_avoidance(
#     cands, od['O'], od['D'],
#     avoid_union, avoid_union_prepared, avoid_union_by_code,
#     verbose=True,
# )

# print(f"\n  실행 시간: {result['total_elapsed_ms']:.1f}ms")
# print(f"  swap 횟수: {result['n_swaps']}")
# print(f"  hard case: {result['hard_case']}")

# print(f"\n  최종 K=3 (방문 순서):")
# sel = result['final_selection']
# for i, hid in enumerate(result['visit_order'], 1):
#     row = sel[sel['hotspot_id'] == hid].iloc[0]
#     print(f"    {i}. {hid}: S_i={row['S_i']:.2f}, "
#           f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}, "
#           f"Score_i={row['Score_i']:.3f}")

# print(f"\n  최종 polyline:")
# print(f"    도보 거리: {result['final_distance_m']/1000:.2f}km")
# print(f"    도보 소요: {result['final_duration_ms']/60000:.1f}분")
# print(f"    점 수: {len(result['final_polyline'])}개")

# fi = result['final_intersection']
# if fi and fi['crosses']:
#     print(f"  회피 영역 교차: {fi['intersection_length_m']:.0f}m "
#           f"({fi['intersection_ratio']*100:.2f}%)")
#     print(f"  UQA 코드: {fi['crossed_uqa_codes']}")
# else:
#     print(f"  회피 영역 교차: 없음")

# if result['swap_history']:
#     print(f"\n  swap 이력:")
#     for h in result['swap_history']:
#         print(f"    #{h['attempt']}: {h['removed']} (p={h['removed_penalty']:.2f}) "
#               f"→ {h['added']}")

In [ ]:
# 셀 11 (수정): 방법 C+swap 통합 알고리즘 + 최선 결과 추적 안전장치

def algorithm_greedy_with_avoidance(candidates_df, origin, destination,
                                      avoid_union_param, avoid_union_prepared_param,
                                      avoid_union_by_code_param,
                                      alpha=ALPHA, beta=BETA, gamma=GAMMA,
                                      k=K, max_swap=MAX_SWAP_ATTEMPTS,
                                      verbose=False):
    """방법 C + swap 통합 알고리즘 (4번째 알고리즘 후보 E_v2).
    
    파이프라인:
    1. Buffer 후보 → S_i Top-15 풀
    2. 방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
    3. Score_i Top-K=3 초기 선정
    4. K=3을 Score_i 내림차순으로 방문
    5. Tmap polyline 호출 + 교차 검사
    6. 교차 시 swap (penalty 큰 후보 → 다음 Score 후보)
    7. ★ 매 attempt마다 best 추적 → 최종적으로 best 반환
    8. 최대 max_swap회까지 반복 후 hard case 종료
    
    수정 (단계 6-α 발견 반영):
    - 최선 결과 추적 (best dict)
    - swap이 결과를 악화시켜도 baseline(또는 더 좋은) 결과 보장
    - 'best_from_attempt' 필드 추가 — 어느 attempt가 best였는지 명시
    
    Returns:
        dict: final_selection, visit_order, final_polyline, final_intersection,
              final_distance_m, final_duration_ms, n_swaps, hard_case,
              swap_history, total_elapsed_ms, best_from_attempt
    """
    t0 = time.time()
    
    # 1. S_i Top-15 풀
    n = len(candidates_df)
    if n == 0:
        return None
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    
    # 2. 방법 C: Score_i 계산
    scored_pool = compute_scores_with_avoidance(
        pool, origin, destination,
        avoid_union_param, avoid_union_prepared_param,
        alpha=alpha, beta=beta, gamma=gamma,
    )
    
    # 3. Score_i Top-K=3 초기 선정
    k_eff = min(k, n)
    initial_selection = scored_pool.nlargest(k_eff, 'Score_i')
    
    # 대체 후보: pool 중 K=3에 없는 것, Score_i 내림차순
    alternates = scored_pool[~scored_pool['hotspot_id'].isin(initial_selection['hotspot_id'])]
    alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
    swap_history = []
    
    # ★ 최선 결과 추적 (NEW)
    best = {
        'cross_m': None,
        'selection': None,
        'visit_order': None,
        'polyline': None,
        'intersection': None,
        'distance_m': 0,
        'duration_ms': 0,
        'attempt_no': None,
    }
    
    if verbose:
        print(f"  초기 K=3 선정 완료. swap 루프 진입.")
    
    for attempt in range(max_swap + 1):
        # 4. K=3 방문 순서 = Score_i 내림차순
        visit_order = current['hotspot_id'].tolist()
        ordered_waypoints = [(row['latitude'], row['longitude'])
                              for _, row in current.iterrows()]
        
        # 5. Tmap polyline 호출
        route = get_pedestrian_route(origin, destination, waypoints=ordered_waypoints)
        if route['status'] != 'OK':
            # API 에러 → best가 있으면 best 반환
            if best['selection'] is not None:
                return {
                    'final_selection': best['selection'],
                    'visit_order': best['visit_order'],
                    'final_polyline': best['polyline'],
                    'final_intersection': best['intersection'],
                    'final_distance_m': best['distance_m'],
                    'final_duration_ms': best['duration_ms'],
                    'n_swaps': attempt,
                    'hard_case': True,
                    'swap_history': swap_history,
                    'total_elapsed_ms': (time.time() - t0) * 1000,
                    'best_from_attempt': best['attempt_no'],
                    'error': route['status'],
                }
            return {
                'final_selection': current, 'visit_order': visit_order,
                'final_polyline': None, 'final_intersection': None,
                'final_distance_m': 0, 'final_duration_ms': 0,
                'n_swaps': attempt, 'hard_case': True,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'error': route['status'],
            }
        
        intersection = check_polyline_avoid_intersection(
            route['path'], avoid_union_param, avoid_union_by_code_param)
        current_cross_m = (intersection['intersection_length_m'] 
                           if intersection['crosses'] else 0)
        
        # ★ best 갱신 (NEW) — 교차 길이가 작을수록 좋음
        if best['cross_m'] is None or current_cross_m < best['cross_m']:
            best['cross_m'] = current_cross_m
            best['selection'] = current.copy()
            best['visit_order'] = visit_order
            best['polyline'] = route['path']
            best['intersection'] = intersection
            best['distance_m'] = route['distance_m']
            best['duration_ms'] = route['duration_ms']
            best['attempt_no'] = attempt
        
        if verbose:
            cross_info = (f"{intersection['intersection_length_m']:.0f}m"
                          if intersection['crosses'] else "없음")
            is_best = " ★best 갱신" if best['attempt_no'] == attempt else ""
            print(f"  attempt {attempt}: 교차 {cross_info}{is_best}")
        
        # 6a. 교차 없으면 성공 종료 (즉시 종료, best도 자동으로 현재)
        if not intersection['crosses']:
            return {
                'final_selection': current,
                'visit_order': visit_order,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': False,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'best_from_attempt': attempt,
            }
        
        # 6b. swap 한도 도달 → ★ best 반환 (NEW)
        if attempt >= max_swap or len(alternates) == 0:
            return {
                'final_selection': best['selection'],
                'visit_order': best['visit_order'],
                'final_polyline': best['polyline'],
                'final_intersection': best['intersection'],
                'final_distance_m': best['distance_m'],
                'final_duration_ms': best['duration_ms'],
                'n_swaps': attempt,
                'hard_case': True,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'best_from_attempt': best['attempt_no'],
            }
        
        # 6c. swap 시도
        worst_idx = current['avoid_penalty_i'].idxmax()
        worst_id = current.loc[worst_idx, 'hotspot_id']
        worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
        new_candidate = alternates.iloc[0]
        alternates = alternates.iloc[1:].reset_index(drop=True)
        
        swap_history.append({
            'attempt': attempt + 1,
            'removed': worst_id,
            'removed_penalty': worst_penalty,
            'added': new_candidate['hotspot_id'],
            'added_penalty': new_candidate['avoid_penalty_i'],
            'intersection_before_m': current_cross_m,
        })
        
        # K=3 갱신
        current = current.drop(index=worst_idx).reset_index(drop=True)
        new_row = new_candidate.to_frame().T
        current = pd.concat([current, new_row], ignore_index=True)
        current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    return None   # 도달 안 함


print("[통합 알고리즘 함수 정의 완료 — 안전장치 추가]")
print(f"  파이프라인: Buffer → S_i Top-{BF_TOP_N} → 방법 C Score_i Top-{K} → Tmap → swap")
print(f"  ★ NEW: 최선 결과 추적. swap이 결과를 악화시킬 수 없음.")
print(f"  하이퍼파라미터: α={ALPHA}, β={BETA}, γ={GAMMA}, K={K}, max_swap={MAX_SWAP_ATTEMPTS}")


# === 단독 동작 검증 — junggu (best 추적 확인) ===
print(f"\n{'=' * 80}")
print(f"[통합 알고리즘 단독 검증 — junggu (best 추적 확인)]")
print(f"{'=' * 80}")

district = 'junggu'
od = OD_CANDIDATES_V2[district]
cands = od_buffer_results[district]['candidates']

result = algorithm_greedy_with_avoidance(
    cands, od['O'], od['D'],
    avoid_union, avoid_union_prepared, avoid_union_by_code,
    verbose=True,
)

print(f"\n  실행 시간: {result['total_elapsed_ms']:.1f}ms")
print(f"  swap 횟수: {result['n_swaps']}")
print(f"  hard case: {result['hard_case']}")
print(f"  ★ best 채택 시점: attempt {result['best_from_attempt']}")

print(f"\n  최종 K=3 (best 시점):")
sel = result['final_selection']
for i, hid in enumerate(result['visit_order'], 1):
    row = sel[sel['hotspot_id'] == hid].iloc[0]
    print(f"    {i}. {hid}: S_i={row['S_i']:.2f}, "
          f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}")

print(f"\n  최종 polyline:")
print(f"    도보 거리: {result['final_distance_m']/1000:.2f}km")
print(f"    회피 교차: {result['final_intersection']['intersection_length_m']:.0f}m"
      if result['final_intersection']['crosses'] else "    회피 교차: 없음")

---

## 단계 6-γ: 현실적 인접 OD 200개 검증 (v4 신규)

여기부터 v4 신규 셀이다. 선행 조건은 위 셀 1~14가 실행되어 `hotspots`, `avoid_union`,
`avoid_union_prepared`, `algorithm_greedy_with_avoidance` 등이 정의된 상태이다.


In [ ]:
# ==========================================================================
# 셀 15: 단계 6-γ 현실적 인접 OD 200개 생성·영속화 (군구 매핑 수정판)
# ==========================================================================
# v3(단계 6-β)의 인접 OD는 회피 폴리곤 중심을 기준으로 양방향에 O·D를 배치하여,
# 면적이 큰 폴리곤에서는 O·D가 모두 폴리곤 내부에 위치하였다. 그 결과 원본 경로의
# 30.9%(중앙값)가 회피 영역 내부를 통과하는 가혹한 표본이 되었고, hard case가
# 70.8%로 측정되었다. 출발지가 회피 영역 내부이면 교차 0은 원리적으로 불가능하다.
#
# v4는 O·D를 모든 회피 폴리곤 외부에 배치하되 OD 직선이 폴리곤을 통과하도록 하여,
# '공단 옆을 지나가는' 실제 사용 시나리오를 재현한다. 이격 거리는 100/300/500m를
# 균등 배분하여 이격에 따른 성능 변화를 함께 관측한다.
# 군구는 균등 순회하여 v3에서 관측된 특정 군구 편중(미추홀 16/48)을 완화한다.

import random
import pickle

# === 단계 6-γ 전용 출력 폴더 ===
V4_OUTPUT_DIR = Path("./output4_260731")
V4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[출력 폴더] {V4_OUTPUT_DIR.resolve()}")

OD_CACHE_V4 = V4_OUTPUT_DIR / "realistic_od_200.pkl"
FORCE_REGEN = False   # True로 두면 캐시가 있어도 새로 생성

# === 생성 파라미터 ===
RANDOM_SEED    = 42
N_TOTAL        = 200                  # 총 OD 수 (전량 현실적 인접군)
MARGIN_GRID    = [100, 300, 500]      # O·D와 회피 영역 경계 간 최소 이격(m), 균등 배분
DIRECT_MIN_KM  = 1.0
DIRECT_MAX_KM  = 4.0
MIN_CANDIDATES = K                    # Buffer 후보 최소 수 (K=3 선정 성립 조건)
INWARD_MIN_M   = 20                   # 폴리곤 내부 기준점의 경계로부터 최소 깊이
INWARD_MAX_M   = 200                  # 동 최대 깊이 (가장자리를 스치는 경로 유도)
RAY_STEP_M     = 40                   # 광선 탐색 간격
RAY_MAX_M      = 3000                 # 광선 탐색 상한 (편도)
MAX_GEN_TRIALS = 60000

DEG_PER_M = 1.0 / 111000.0            # 셀 1~14와 동일한 111km/도 근사 유지

# === 회피 폴리곤을 군구별로 그룹화 ===
# 셀 3에서 정의한 match_district()를 사용한다. 부분 문자열 충돌('동구' ⊂ '남동구')로
# 남동구가 동구로 오분류되던 v4(260729) 버그를 해소한 매핑이다.
poly_by_district = defaultdict(list)
n_unmatched_poly = 0
for p in avoid_polygons:
    d_eng = match_district(p['sigg_name'])
    if d_eng is None:
        n_unmatched_poly += 1
        continue
    poly_by_district[d_eng].append(p)

print(f"[폴리곤 군구 분류] {len(poly_by_district)}개 군구, 미매칭 {n_unmatched_poly}개")
for d in sorted(poly_by_district, key=lambda k: -len(poly_by_district[k])):
    n_hs = len(hotspots[hotspots['source_district'] == d])
    print(f"    {d:<10} 폴리곤 {len(poly_by_district[d]):>3}개 | 핫스팟 {n_hs:>4}곳")

# 핫스팟이 존재하는 군구만 대상 (Buffer 후보 확보 가능 조건)
districts_target = [d for d in poly_by_district
                    if len(hotspots[hotspots['source_district'] == d]) >= MIN_CANDIDATES]
print(f"[대상 군구] {len(districts_target)}개: {districts_target}")

# === 이격 거리별 버퍼 유니온 사전 계산 (O·D 유효성 판정용) ===
# 매 시도마다 avoid_union.distance()를 호출하면 비용이 크므로,
# 이격 거리별로 확장한 유니온을 1회 생성하고 prepared로 포함 판정만 수행한다.
print("[버퍼 유니온 사전 계산]", end=" ")
BUFFERED = {}
for m in MARGIN_GRID:
    g = avoid_union.buffer(m * DEG_PER_M)
    BUFFERED[m] = prep(g)
    print(f"{m}m", end=" ")
print("완료")


def _sample_inner_point(poly):
    """폴리곤 내부에서 경계에 가까운 기준점을 추출한다 (가장자리 통과 경로 유도)."""
    depth = np.random.uniform(INWARD_MIN_M, INWARD_MAX_M) * DEG_PER_M
    shrunk = poly.buffer(-depth)
    if shrunk.is_empty:                      # 소형 폴리곤은 축소 시 소멸
        return poly.centroid
    if shrunk.geom_type == 'MultiPolygon':
        shrunk = max(shrunk.geoms, key=lambda g: g.area)
    ring = shrunk.exterior
    return ring.interpolate(np.random.uniform(0, 1), normalized=True)


def _walk_out(origin_pt, theta, margin_m):
    """기준점에서 theta 방향으로 진행하며 버퍼 유니온을 벗어나는 첫 지점을 반환한다."""
    prepared = BUFFERED[margin_m]
    lat0 = origin_pt.y
    coslat = math.cos(math.radians(lat0))
    steps = int(RAY_MAX_M / RAY_STEP_M)
    for i in range(1, steps + 1):
        d_m = i * RAY_STEP_M
        dlat = d_m * DEG_PER_M * math.sin(theta)
        dlon = d_m * DEG_PER_M * math.cos(theta) / coslat
        pt = Point(origin_pt.x + dlon, origin_pt.y + dlat)
        if not prepared.contains(pt):
            return (pt.y, pt.x)              # (lat, lon)
    return None                              # 상한 내 이탈 실패


def _buffer_candidates(O, D, d_eng):
    sub = hotspots[hotspots['source_district'] == d_eng]
    if len(sub) == 0:
        return sub
    return filter_candidates_by_ellipse_buffer(sub, O, D, get_buffer_radius(d_eng))


def _build_realistic_od():
    """O·D는 모든 회피 폴리곤 외부, OD 직선은 폴리곤 통과인 표본을 생성한다."""
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    generated = []
    trials = 0
    d_idx = 0                                # 군구 균등 순회 인덱스

    while len(generated) < N_TOTAL and trials < MAX_GEN_TRIALS:
        trials += 1

        d_eng = districts_target[d_idx % len(districts_target)]
        margin_m = MARGIN_GRID[len(generated) % len(MARGIN_GRID)]   # 이격 균등 배분

        polys = poly_by_district[d_eng]
        if not polys:
            d_idx += 1
            continue
        poly = polys[np.random.randint(len(polys))]['geom']
        if poly.geom_type == 'MultiPolygon':
            poly = max(poly.geoms, key=lambda g: g.area)

        P = _sample_inner_point(poly)
        theta = np.random.uniform(0, 2 * math.pi)

        D_pt = _walk_out(P, theta, margin_m)
        O_pt = _walk_out(P, theta + math.pi, margin_m)
        if O_pt is None or D_pt is None:
            continue

        O, D = O_pt, D_pt
        direct_km = haversine_km(O, D)
        if not (DIRECT_MIN_KM <= direct_km <= DIRECT_MAX_KM):
            continue

        # 검증 1: O·D가 모든 회피 폴리곤 외부이며 이격 조건 충족
        if BUFFERED[margin_m].contains(Point(O[1], O[0])):
            continue
        if BUFFERED[margin_m].contains(Point(D[1], D[0])):
            continue

        # 검증 2: OD 직선이 회피 영역을 실제로 통과
        line = LineString([(O[1], O[0]), (D[1], D[0])])
        if not line.intersects(avoid_union):
            continue
        chord_m = line.intersection(avoid_union).length * 111000.0
        if chord_m < 1.0:                    # 접점 수준은 제외
            continue

        cands = _buffer_candidates(O, D, d_eng)
        if len(cands) < MIN_CANDIDATES:
            continue

        od_margin_m = min(avoid_union.distance(Point(O[1], O[0])),
                          avoid_union.distance(Point(D[1], D[0]))) * 111000.0
        n_avoid_in_buffer = int(cands['in_avoid_zone'].sum()) if 'in_avoid_zone' in cands.columns else 0

        generated.append({
            'od_id': f"V{len(generated):03d}",
            'O': O, 'D': D,
            'district': d_eng,
            'margin_target_m': margin_m,
            'od_margin_m': od_margin_m,
            'direct_km': direct_km,
            'chord_m': chord_m,
            'n_candidates': len(cands),
            'n_avoid_in_buffer': n_avoid_in_buffer,
            'candidates': cands,
        })
        d_idx += 1

    return generated, trials


# === 로드 우선, 없으면 생성·저장 ===
print("=" * 90)
print("[단계 6-gamma 현실적 인접 OD 200개 — O·D 회피영역 외부 + 직선 통과]")
print("=" * 90)

if OD_CACHE_V4.exists() and not FORCE_REGEN:
    with open(OD_CACHE_V4, "rb") as f:
        REALISTIC_OD = pickle.load(f)
    print(f"\n  [OD 로드] {OD_CACHE_V4.name} 재사용 — {len(REALISTIC_OD)}개")
    print(f"    (새로 생성하려면 FORCE_REGEN=True 또는 pkl 파일 삭제 후 재실행)")
else:
    t0 = time.time()
    REALISTIC_OD, trials = _build_realistic_od()
    with open(OD_CACHE_V4, "wb") as f:
        pickle.dump(REALISTIC_OD, f)
    print(f"\n  [OD 생성·저장] 시도 {trials}회 -> {len(REALISTIC_OD)}개 ({time.time()-t0:.0f}s)")
    print(f"    저장 위치: {OD_CACHE_V4.resolve()}")
    if len(REALISTIC_OD) < N_TOTAL:
        print(f"    [경고] 목표 미달. RAY_MAX_M 상향 또는 DIRECT_MAX_KM 완화 필요.")

# === 생성 조건 자동 검증 (v3에서 누락되어 표본 문제를 사후 발견한 항목) ===
print("\n" + "-" * 90)
print("[생성 조건 자동 검증]")
n_O_inside = sum(1 for od in REALISTIC_OD if avoid_union_prepared.contains(Point(od['O'][1], od['O'][0])))
n_D_inside = sum(1 for od in REALISTIC_OD if avoid_union_prepared.contains(Point(od['D'][1], od['D'][0])))
n_no_cross = sum(1 for od in REALISTIC_OD
                 if not LineString([(od['O'][1], od['O'][0]), (od['D'][1], od['D'][0])]).intersects(avoid_union))
n_margin_bad = sum(1 for od in REALISTIC_OD if od['od_margin_m'] < od['margin_target_m'] * 0.95)
print(f"  O가 회피영역 내부인 OD: {n_O_inside}개   (기대값 0)")
print(f"  D가 회피영역 내부인 OD: {n_D_inside}개   (기대값 0)")
print(f"  직선이 회피영역을 통과하지 않는 OD: {n_no_cross}개   (기대값 0)")
print(f"  목표 이격 미달 OD: {n_margin_bad}개   (기대값 0)")
if n_O_inside or n_D_inside or n_no_cross or n_margin_bad:
    print("  [경고] 생성 조건 위반이 존재한다. 표본을 재생성해야 한다.")
else:
    print("  모든 조건 충족. v3 대비 표본 현실성이 확보되었다.")

# === 표본 요약 ===
df_od4 = pd.DataFrame([{k: v for k, v in od.items() if k != 'candidates'} for od in REALISTIC_OD])
if len(df_od4) > 0:
    print("\n" + "-" * 90)
    print("[표본 요약]")
    print(f"  직선 거리(km): 평균 {df_od4['direct_km'].mean():.2f} "
          f"[{df_od4['direct_km'].min():.2f}, {df_od4['direct_km'].max():.2f}]")
    print(f"  OD 직선의 회피영역 통과 길이(m): 중앙값 {df_od4['chord_m'].median():.0f} "
          f"[{df_od4['chord_m'].min():.0f}, {df_od4['chord_m'].max():.0f}]")
    print(f"  실제 O·D 이격(m): 중앙값 {df_od4['od_margin_m'].median():.0f}")
    print(f"  Buffer 후보 수: 평균 {df_od4['n_candidates'].mean():.1f} "
          f"[{df_od4['n_candidates'].min()}, {df_od4['n_candidates'].max()}]")
    print(f"  Buffer 안 회피 핫스팟: OD 평균 {df_od4['n_avoid_in_buffer'].mean():.2f}곳")
    print("\n  이격 목표별 분포:")
    print(df_od4.groupby('margin_target_m').agg(
        OD수=('od_id', 'count'), 직선km평균=('direct_km', 'mean'),
        통과길이m중앙=('chord_m', 'median')).round(2).to_string())
    print("\n  군구별 분포:")
    print(df_od4['district'].value_counts().to_string())


In [ ]:
# ==========================================================================
# 셀 16: 단계 6-γ 배치 실행 (γ=1.0 단일, 25개×8배치, checkpoint·resume·병합)
# ==========================================================================
# v3에서 γ 0.5~5.0 구간의 조정 효과가 낮음을 확인하였다(인접 48쌍 중 41쌍이 무반응,
# 총 감소율 차이 2.6%p). 이에 v4는 γ=1.0으로 고정하여 OD당 Tmap 호출을 1/4로 줄이고,
# 절감분을 표본 규모 확대(100 -> 200)에 배분한다.
# 배치 checkpoint·resume·Tmap 응답 캐시 구조는 v3에서 실환경 검증된 방식을 계승한다.

GAMMA_FIXED = 1.0
COORD_ROUND = 6      # 좌표 캐시 키 반올림 자리수 (약 0.1m 해상도)
BATCH_SIZE  = 25     # 200 OD -> 8배치
FORCE_RERUN = False  # True로 두면 기존 배치 checkpoint를 삭제하고 처음부터 재실행

# === Tmap 응답 캐시 (비침습적 monkey-patch) ===
if not getattr(get_pedestrian_route, '_is_cache_wrapper', False):
    _get_pedestrian_route_raw = get_pedestrian_route
    _tmap_cache = {}
    _tmap_stats = {'hit': 0, 'miss': 0}

    def get_pedestrian_route(origin, destination, waypoints=None, debug=False):
        key = (
            round(origin[0], COORD_ROUND), round(origin[1], COORD_ROUND),
            round(destination[0], COORD_ROUND), round(destination[1], COORD_ROUND),
            tuple((round(w[0], COORD_ROUND), round(w[1], COORD_ROUND))
                  for w in (waypoints or [])),
        )
        if key in _tmap_cache:
            _tmap_stats['hit'] += 1
            return _tmap_cache[key]
        _tmap_stats['miss'] += 1
        res = _get_pedestrian_route_raw(origin, destination, waypoints=waypoints, debug=debug)
        if res.get('status') == 'OK':
            _tmap_cache[key] = res
        return res

    get_pedestrian_route._is_cache_wrapper = True
    print("[Tmap 응답 캐시 적용] get_pedestrian_route 래핑 완료")
else:
    _tmap_cache.clear()
    _tmap_stats['hit'] = 0
    _tmap_stats['miss'] = 0
    print("[Tmap 응답 캐시 초기화] 기존 래퍼 재사용")


def _run_one_od_v4(od):
    """OD 1개: 원본 측정 + γ=1.0 방법 C+swap 실행 -> 결과 행(dict) 또는 None."""
    sel_o, _ = algorithm_greedy_original(od['candidates'], od['O'], od['D'])
    if len(sel_o) == 0:
        return None
    wp = [(r['latitude'], r['longitude']) for _, r in sel_o.iterrows()]
    route_o = get_pedestrian_route(od['O'], od['D'], waypoints=wp)
    if route_o['status'] != 'OK':
        return None
    inter_o = check_polyline_avoid_intersection(route_o['path'], avoid_union, avoid_union_by_code)
    orig_cross_m = inter_o['intersection_length_m'] if inter_o['crosses'] else 0.0
    orig_dist_km = route_o['distance_m'] / 1000

    res = algorithm_greedy_with_avoidance(
        od['candidates'], od['O'], od['D'],
        avoid_union, avoid_union_prepared, avoid_union_by_code,
        gamma=GAMMA_FIXED,
    )
    if res is None or res.get('final_polyline') is None:
        return None
    fi = res['final_intersection']
    v2_cross_m = fi['intersection_length_m'] if (fi and fi['crosses']) else 0.0
    v2_dist_km = res['final_distance_m'] / 1000

    return {
        'od_id': od['od_id'], 'district': od['district'],
        'margin_target_m': od['margin_target_m'], 'od_margin_m': od['od_margin_m'],
        'direct_km': od['direct_km'], 'chord_m': od['chord_m'],
        'n_candidates': od['n_candidates'], 'n_avoid_in_buffer': od['n_avoid_in_buffer'],
        'gamma': GAMMA_FIXED,
        'orig_cross_m': orig_cross_m, 'v2_cross_m': v2_cross_m,
        'diff_m': v2_cross_m - orig_cross_m,
        'orig_dist_km': orig_dist_km, 'v2_dist_km': v2_dist_km,
        'dist_delta_km': v2_dist_km - orig_dist_km,
        # v3에서 사후 계산해야 했던 지표를 처음부터 기록한다
        'orig_cross_ratio_pct': orig_cross_m / (orig_dist_km * 1000) * 100 if orig_dist_km > 0 else 0.0,
        'v2_cross_ratio_pct': v2_cross_m / (v2_dist_km * 1000) * 100 if v2_dist_km > 0 else 0.0,
        'n_swaps': res['n_swaps'], 'hard_case': bool(res['hard_case']),
        'best_from_attempt': res.get('best_from_attempt'),
    }


n_batches = (len(REALISTIC_OD) + BATCH_SIZE - 1) // BATCH_SIZE

print("=" * 90)
print(f"[단계 6-gamma 배치 실행 — {len(REALISTIC_OD)} OD / 배치크기 {BATCH_SIZE} / "
      f"{n_batches}배치 / γ={GAMMA_FIXED} 고정]")
print("=" * 90)

if FORCE_RERUN:
    for b in range(n_batches):
        ck = V4_OUTPUT_DIR / f"step6_gamma_sweep_batch{b}.csv"
        if ck.exists():
            ck.unlink()
    print("[FORCE_RERUN] 기존 배치 checkpoint 전체 삭제")

t_start = time.time()
for b in range(n_batches):
    ckpt = V4_OUTPUT_DIR / f"step6_gamma_sweep_batch{b}.csv"
    if ckpt.exists():
        print(f"  배치 {b} 건너뜀 (checkpoint 존재: {ckpt.name})")
        continue

    batch_ods = REALISTIC_OD[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    rows, n_fail = [], 0
    for od in batch_ods:
        r = _run_one_od_v4(od)
        if r is None:
            n_fail += 1
            continue
        rows.append(r)

    pd.DataFrame(rows).to_csv(ckpt, index=False, encoding='utf-8-sig')
    print(f"  배치 {b} 완료: OD {len(batch_ods)}개, 성공 {len(rows)}, 실패 {n_fail} "
          f"| 경과 {time.time()-t_start:.0f}s "
          f"| Tmap 호출 {_tmap_stats['miss']} (적중 {_tmap_stats['hit']})")

batch_files = [V4_OUTPUT_DIR / f"step6_gamma_sweep_batch{b}.csv" for b in range(n_batches)]
df_v4 = pd.concat([pd.read_csv(f) for f in batch_files if f.exists()], ignore_index=True)
v4_csv = V4_OUTPUT_DIR / 'step6_gamma_200od_realistic.csv'
df_v4.to_csv(v4_csv, index=False, encoding='utf-8-sig')

n_calls = _tmap_stats['hit'] + _tmap_stats['miss']
print(f"\n[병합 완료] {len(df_v4)}행 (OD 1행) -> {v4_csv.resolve()}")
if n_calls:
    print(f"  Tmap 실제 호출 {_tmap_stats['miss']}회 / 캐시 적중 {_tmap_stats['hit']}회 "
          f"(절감율 {_tmap_stats['hit']/n_calls*100:.1f}%)")
else:
    print("  (호출 없음 — 전 배치 건너뜀)")


In [ ]:
# ==========================================================================
# 셀 17: 단계 6-γ 집계 + v3 대비 비교 + 환경 변수 재산출
# ==========================================================================
# v3(가혹 조건)와 v4(현실 조건)를 나란히 제시하여 알고리즘 성능 범위를 규정한다.
# ALGO_HARD_CASE_M은 표본에 따라 크게 달라지므로 v4 분포로 재산출한다.

# === v3 참조값 (avoidZoneTest_v3_260727 인접군 48쌍, γ=1.0) ===
V3_REF = {
    'n': 48, 'hard_case_pct': 70.83, 'total_reduction_pct': 20.52,
    'residual_median_m': 1522.0, 'dist_delta_mean_km': -0.22,
    'orig_cross_ratio_median_pct': 30.9, 'hard_case_m': 848,
}


def _stats_v4(df):
    crossing = df[df['orig_cross_m'] > 0]
    if len(crossing) > 0:
        tot = (crossing['orig_cross_m'].sum() - crossing['v2_cross_m'].sum()) / crossing['orig_cross_m'].sum() * 100
        mean_red = (((crossing['orig_cross_m'] - crossing['v2_cross_m']) / crossing['orig_cross_m']) * 100).mean()
        full = int(((crossing['orig_cross_m'] - crossing['v2_cross_m']) / crossing['orig_cross_m'] > 0.999).sum())
    else:
        tot = mean_red = 0.0
        full = 0
    return {
        'n': len(df),
        'n_orig_crossing': int((df['orig_cross_m'] > 0).sum()),
        'crossing_pct': (df['orig_cross_m'] > 0).mean() * 100,
        'mean_swaps': df['n_swaps'].mean(),
        'hard_case_pct': df['hard_case'].mean() * 100,
        'total_reduction_pct': tot,
        'mean_reduction_pct': mean_red,
        'n_full_removed': full,
        'n_worse_vs_orig': int((df['diff_m'] > 1).sum()),
        'mean_dist_delta_km': df['dist_delta_km'].mean(),
        'residual_median_m': df[df['v2_cross_m'] > 0]['v2_cross_m'].median() if (df['v2_cross_m'] > 0).any() else 0.0,
    }

pd.set_option('display.float_format', lambda x: f"{x:.2f}")

print("=" * 100)
print("[단계 6-gamma 집계 — 이격 거리별 / 전체]")
print("=" * 100)

rows = []
for m in sorted(df_v4['margin_target_m'].unique()):
    s = _stats_v4(df_v4[df_v4['margin_target_m'] == m]); s['margin_m'] = m
    rows.append(s)
s = _stats_v4(df_v4); s['margin_m'] = 'ALL'
rows.append(s)
df_sum4 = pd.DataFrame(rows)[
    ['margin_m', 'n', 'n_orig_crossing', 'crossing_pct', 'mean_swaps', 'hard_case_pct',
     'total_reduction_pct', 'mean_reduction_pct', 'n_full_removed', 'n_worse_vs_orig',
     'mean_dist_delta_km', 'residual_median_m']
]
print(df_sum4.to_string(index=False))

# === 표본 현실성 검증: 경로 중 회피영역 통과 비율 ===
cross4 = df_v4[df_v4['orig_cross_m'] > 0]
ratio_med = cross4['orig_cross_ratio_pct'].median() if len(cross4) else 0.0
print("\n" + "-" * 100)
print("[표본 현실성 — 원본 경로 중 회피영역 통과 비율]")
print(f"  v4 중앙값 {ratio_med:.1f}%   (v3 {V3_REF['orig_cross_ratio_median_pct']:.1f}%)")
print(f"  통과 비율 50% 초과 OD: {int((cross4['orig_cross_ratio_pct'] > 50).sum())}개 "
      f"/ 교차 발생 {len(cross4)}개")

# === v3 대비 비교 ===
allv4 = df_sum4[df_sum4['margin_m'] == 'ALL'].iloc[0]
print("\n" + "=" * 100)
print("[v3(가혹 조건) vs v4(현실 조건) 비교]")
print("=" * 100)
cmp_rows = [
    ('표본 수 (OD)',            f"{V3_REF['n']}",                              f"{int(allv4['n'])}"),
    ('hard case 비율 (%)',      f"{V3_REF['hard_case_pct']:.1f}",              f"{allv4['hard_case_pct']:.1f}"),
    ('총 교차 감소율 (%)',      f"{V3_REF['total_reduction_pct']:.1f}",        f"{allv4['total_reduction_pct']:.1f}"),
    ('잔존 교차 중앙값 (m)',    f"{V3_REF['residual_median_m']:.0f}",          f"{allv4['residual_median_m']:.0f}"),
    ('평균 도보 거리Δ (km)',    f"{V3_REF['dist_delta_mean_km']:+.2f}",        f"{allv4['mean_dist_delta_km']:+.2f}"),
    ('경로 중 통과 비율 (%)',   f"{V3_REF['orig_cross_ratio_median_pct']:.1f}", f"{ratio_med:.1f}"),
]
print(f"  {'지표':<24}{'v3 (가혹)':>14}{'v4 (현실)':>14}")
for name, a, b in cmp_rows:
    print(f"  {name:<24}{a:>14}{b:>14}")

# === 환경 변수 재산출 ===
bfa4 = df_v4[df_v4['orig_cross_m'] > 0]['best_from_attempt'].dropna()
p95_4 = int(np.ceil(np.percentile(bfa4, 95))) if len(bfa4) else 0
rec_max_swap_v4 = max(p95_4, 2) if len(bfa4) else MAX_SWAP_ATTEMPTS

resid4 = df_v4[df_v4['v2_cross_m'] > 0]['v2_cross_m']
if len(resid4) > 0:
    p25_4 = float(np.percentile(resid4, 25))
    med_4 = float(np.percentile(resid4, 50))
    rec_hard_case_m_v4 = int(max(p25_4, 50))    # 상한 없음 (v3 7-1절 결함 정정 반영)
else:
    p25_4 = med_4 = 0.0
    rec_hard_case_m_v4 = V3_REF['hard_case_m']

print("\n[best 달성 attempt 분포 — 교차 발생 OD]")
print(f"  표본 {len(bfa4)} | 95퍼센타일 {p95_4} | 최대 {int(bfa4.max()) if len(bfa4) else 0}")
print("[방법 C+swap 잔존 교차(>0m) 분포]")
print(f"  표본 {len(resid4)} | 25% {p25_4:.0f}m | 중앙값 {med_4:.0f}m   (v3: 25% 848m / 중앙 1568m)")

print("\n" + "=" * 100)
print("[환경 변수 권장값 — v4 현실 조건 기준]")
print("=" * 100)
print("\n" + "\n".join([
    f"ALGO_ALPHA={ALPHA}",
    f"ALGO_BETA={BETA}",
    f"ALGO_GAMMA={GAMMA_FIXED}",
    f"ALGO_K={K}",
    f"ALGO_BF_TOP_N={BF_TOP_N}",
    f"ALGO_MAX_SWAP={rec_max_swap_v4}",
    f"ALGO_HARD_CASE_M={rec_hard_case_m_v4}",
]))
print("\n[v3 대비 변경 사항]")
print(f"  ALGO_MAX_SWAP: {4} -> {rec_max_swap_v4}   (현실 조건 95퍼센타일 attempt={p95_4})")
print(f"  ALGO_HARD_CASE_M: {V3_REF['hard_case_m']} -> {rec_hard_case_m_v4}   "
      f"(현실 조건 잔존 교차 25퍼센타일 {p25_4:.0f}m)")
print("  ALGO_GAMMA: 1.0 유지 (v3에서 확정, v4는 단일 γ로 실행)")

sum4_csv = V4_OUTPUT_DIR / 'step6_gamma_summary.csv'
df_sum4.to_csv(sum4_csv, index=False, encoding='utf-8-sig')
print(f"\n[CSV 저장] {sum4_csv.resolve()}")
pd.reset_option('display.float_format')


### 시각화

셀 17 실행 후 진행한다. 셀 18은 차트 6종 PNG를, 셀 19는 대표 OD 경로 지도 HTML을
`output4_260729`에 저장한다. matplotlib 한글 폰트는 Windows 기준 `Malgun Gothic`으로
지정되어 있으며, 다른 OS에서는 셀 18 상단의 폰트명을 교체한다.


In [ ]:
# ==========================================================================
# 셀 18: 단계 6-γ 시각화 (1) — 정적 차트 6종
# ==========================================================================
# 셀 17이 산출한 df_v4 / df_sum4 / rec_hard_case_m_v4 에 의존한다.

import matplotlib
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows 기준
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 3, figsize=(19, 10))

# (1) 이격 거리별 hard case 비율 / 총 감소율
ax = axes[0, 0]
sub = df_sum4[df_sum4['margin_m'] != 'ALL']
x = np.arange(len(sub)); w = 0.35
ax.bar(x - w/2, sub['hard_case_pct'], w, label='hard case 비율', color='#c0504d')
ax.bar(x + w/2, sub['total_reduction_pct'], w, label='총 교차 감소율', color='#4f81bd')
ax.set_xticks(x); ax.set_xticklabels([f"{int(m)}m" for m in sub['margin_m']])
ax.set_xlabel('O·D 이격 거리'); ax.set_ylabel('%')
ax.set_title('(1) 이격 거리별 성능'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# (2) v3 vs v4 핵심 지표 비교
ax = axes[0, 1]
a4 = df_sum4[df_sum4['margin_m'] == 'ALL'].iloc[0]
labels = ['hard case\n비율(%)', '총 교차\n감소율(%)', '경로 중\n통과비율(%)']
v3v = [V3_REF['hard_case_pct'], V3_REF['total_reduction_pct'], V3_REF['orig_cross_ratio_median_pct']]
v4v = [a4['hard_case_pct'], a4['total_reduction_pct'], ratio_med]
x = np.arange(len(labels)); w = 0.35
ax.bar(x - w/2, v3v, w, label='v3 (가혹)', color='#8064a2')
ax.bar(x + w/2, v4v, w, label='v4 (현실)', color='#9bbb59')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title('(2) v3 vs v4 비교'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# (3) 원본 대비 페어드 산점도
ax = axes[0, 2]
ax.scatter(df_v4['orig_cross_m'], df_v4['v2_cross_m'], s=26, alpha=0.6, color='#4f81bd', zorder=3)
lim = max(df_v4['orig_cross_m'].max(), df_v4['v2_cross_m'].max(), 1.0) * 1.05
ax.plot([0, lim], [0, lim], 'r--', alpha=0.7, label='y=x (동일)')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('원본 교차 (m)'); ax.set_ylabel('방법 C+swap 교차 (m)')
ax.set_title('(3) 원본 대비 페어드 — 대각선 위=악화'); ax.legend(); ax.grid(alpha=0.3)

# (4) 잔존 교차 히스토그램 + 임계선
ax = axes[1, 0]
r4 = df_v4[df_v4['v2_cross_m'] > 0]['v2_cross_m']
if len(r4) > 0:
    ax.hist(r4, bins=20, color='#8064a2', alpha=0.75)
ax.axvline(rec_hard_case_m_v4, color='red', ls='--', label=f'v4 권장 {rec_hard_case_m_v4}m')
ax.axvline(V3_REF['hard_case_m'], color='gray', ls=':', label=f"v3 {V3_REF['hard_case_m']}m")
ax.set_xlabel('잔존 교차 (m)'); ax.set_ylabel('빈도')
ax.set_title('(4) 잔존 교차 분포'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# (5) 경로 중 통과 비율 분포 (표본 현실성)
ax = axes[1, 1]
cr = df_v4[df_v4['orig_cross_m'] > 0]['orig_cross_ratio_pct']
if len(cr) > 0:
    ax.hist(cr, bins=20, color='#4bacc6', alpha=0.8)
ax.axvline(ratio_med, color='green', ls='--', label=f'v4 중앙값 {ratio_med:.1f}%')
ax.axvline(V3_REF['orig_cross_ratio_median_pct'], color='red', ls=':',
           label=f"v3 중앙값 {V3_REF['orig_cross_ratio_median_pct']:.1f}%")
ax.set_xlabel('원본 경로 중 회피영역 통과 비율 (%)'); ax.set_ylabel('빈도')
ax.set_title('(5) 표본 현실성 검증'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# (6) 군구별 hard case 비율
ax = axes[1, 2]
gd = df_v4.groupby('district').agg(n=('od_id', 'count'), hard=('hard_case', 'mean'))
gd['hard'] *= 100
gd = gd.sort_values('hard', ascending=False)
ax.barh(gd.index, gd['hard'], color='#f79646')
ax.set_xlabel('hard case 비율 (%)')
ax.set_title('(6) 군구별 hard case'); ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

fig.tight_layout()
chart4_png = V4_OUTPUT_DIR / 'step6_gamma_charts.png'
fig.savefig(chart4_png, dpi=120, bbox_inches='tight')
print(f"[차트 저장] {chart4_png.resolve()}")
plt.show()


In [ ]:
# ==========================================================================
# 셀 19: 단계 6-γ 시각화 (2) — 대표 OD 경로 지도 (folium)
# ==========================================================================
# 교차 감소량이 가장 큰 OD를 대표로 선정하여, 회피 영역·원본 경로·방법 C+swap
# 경로를 비교 시각화한다. O·D가 회피 영역 외부에 위치함을 지도로 확인할 수 있다.

_cand4 = df_v4[df_v4['orig_cross_m'] > 0].copy()

if len(_cand4) == 0:
    print("[지도 생략] 교차 발생 OD가 없어 대표 OD를 선정할 수 없다.")
    m4 = None
else:
    _cand4['reduction_m'] = _cand4['orig_cross_m'] - _cand4['v2_cross_m']
    rep_id4 = _cand4.sort_values('reduction_m', ascending=False).iloc[0]['od_id']
    rep_od4 = next(od for od in REALISTIC_OD if od['od_id'] == rep_id4)
    O, D, cands = rep_od4['O'], rep_od4['D'], rep_od4['candidates']
    print(f"[대표 OD] {rep_id4} (군구 {rep_od4['district']}, 직선 {rep_od4['direct_km']:.2f}km, "
          f"이격 목표 {rep_od4['margin_target_m']}m)")

    sel_o, _ = algorithm_greedy_original(cands, O, D)
    wp_o = [(r['latitude'], r['longitude']) for _, r in sel_o.iterrows()]
    route_o = get_pedestrian_route(O, D, waypoints=wp_o)
    path_o = route_o['path'] if route_o['status'] == 'OK' else []

    res4 = algorithm_greedy_with_avoidance(
        cands, O, D, avoid_union, avoid_union_prepared, avoid_union_by_code,
        gamma=GAMMA_FIXED)
    path_v2 = res4.get('final_polyline') or []
    sel_v2 = res4['final_selection']

    center = [(O[0] + D[0]) / 2, (O[1] + D[1]) / 2]
    m4 = folium.Map(location=center, zoom_start=14, tiles='CartoDB positron')

    lat_lo, lat_hi = min(O[0], D[0]) - 0.02, max(O[0], D[0]) + 0.02
    lon_lo, lon_hi = min(O[1], D[1]) - 0.02, max(O[1], D[1]) + 0.02
    for p in avoid_polygons:
        minx, miny, maxx, maxy = p['geom'].bounds
        if maxx < lon_lo or minx > lon_hi or maxy < lat_lo or miny > lat_hi:
            continue
        geom = p['geom']
        polys = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
        for poly in polys:
            coords = [[y, x] for x, y in poly.exterior.coords]
            folium.Polygon(coords, color='#d9534f', weight=1, fill=True,
                           fill_opacity=0.25, popup=str(p.get('uname', ''))).add_to(m4)

    if path_o:
        folium.PolyLine([[la, lo] for la, lo in path_o], color='#7f7f7f',
                        weight=4, opacity=0.9, popup='원본 경로').add_to(m4)
    if path_v2:
        folium.PolyLine([[la, lo] for la, lo in path_v2], color='#2b7bba',
                        weight=4, opacity=0.9, popup=f'방법 C+swap (γ={GAMMA_FIXED})').add_to(m4)

    folium.Marker([O[0], O[1]], popup='출발(O)', icon=folium.Icon(color='green')).add_to(m4)
    folium.Marker([D[0], D[1]], popup='도착(D)', icon=folium.Icon(color='red')).add_to(m4)
    for _, r in sel_v2.iterrows():
        folium.CircleMarker([r['latitude'], r['longitude']], radius=5, color='#2b7bba',
                            fill=True, fill_opacity=0.9,
                            popup=f"{r['hotspot_id']} S_i={r['S_i']:.1f}").add_to(m4)

    map4_html = V4_OUTPUT_DIR / f'step6_gamma_rep_route_{rep_id4}.html'
    m4.save(str(map4_html))
    print(f"[지도 저장] {map4_html.resolve()}")

m4   # 노트북 인라인 표시
